# Etapa 2: Análise e Tratamento de Valores Ausentes (NaN / NA)

Em Machine Learning, a maioria dos modelos matemáticos (como Regressão Linear, Random Forest e Redes Neurais) **não aceita dados ausentes** (valores nulos, `NaN` ou vazios).

Nesta etapa, vamos:
1. Mapear todas as colunas com dados ausentes nos conjuntos de **Treino** e de **Teste**.
2. Entender a diferença fundamental entre **dado faltante por omissão** e **ausência estrutural** (ex.: a casa simplesmente não possui piscina ou lareira).
3. Definir a estratégia correta de tratamento para cada caso.

In [1]:
import pandas as pd
import numpy as np

# 1. Carregando os dados
df_train = pd.read_csv("train.csv")
df_test = pd.read_csv("test.csv")

# 2. Identificando colunas com valores nulos no treino
nulos_treino = df_train.isnull().sum()
nulos_treino = nulos_treino[nulos_treino > 0]
perc_treino = (nulos_treino / len(df_train)) * 100

tabela_treino = pd.DataFrame({
    "Total Nulos": nulos_treino,
    "Percentual (%)": perc_treino.round(2)
}).sort_values(by="Total Nulos", ascending=False)

print(f"Total de colunas com nulos no Treino: {len(tabela_treino)}")
tabela_treino.head(10)

Total de colunas com nulos no Treino: 19


,Total Nulos,Percentual (%)
PoolQC,1453,99.52
MiscFeature,1406,96.30
Alley,1369,93.77
Fence,1179,80.75
MasVnrType,872,59.73
FireplaceQu,690,47.26
LotFrontage,259,17.74
GarageType,81,5.55
GarageYrBlt,81,5.55
GarageFinish,81,5.55


### 2.1 Alerta Crítico de Machine Learning: Treino vs Teste
Ao inspecionarmos o conjunto de dados, descobrimos um fato essencial:
- O **Treino** possui **19 colunas** com dados nulos.
- O **Teste** possui **33 colunas** com dados nulos!

Ou seja, no conjunto de teste existem **14 colunas adicionais** que contêm valores faltantes esporádicos (como `MSZoning`, `Utilities`, `Exterior1st`, `TotalBsmtSF`, `KitchenQual`, etc.).

> **Lição Fundamental:** Um pipeline profissional de Machine Learning precisa tratar adequadamente tanto os nulos do treino quanto os nulos do teste para que o modelo não falhe na hora de prever novos dados.

In [2]:
# Comparando nulos do Treino e do Teste lado a lado
nulos_teste = df_test.isnull().sum()
nulos_teste = nulos_teste[nulos_teste > 0]
perc_teste = (nulos_teste / len(df_test)) * 100

tabela_teste = pd.DataFrame({
    "Total Nulos Teste": nulos_teste,
    "Percentual Teste (%)": perc_teste.round(2)
})

# Combinando as visões para análise completa
comparativo = pd.concat([tabela_treino, tabela_teste], axis=1).fillna(0)
comparativo.columns = ["Nulos Treino", "% Treino", "Nulos Teste", "% Teste"]
comparativo = comparativo.sort_values(by=["Nulos Treino", "Nulos Teste"], ascending=False)

print(f"Total de colunas únicas com algum dado ausente: {len(comparativo)}")
comparativo

Total de colunas únicas com algum dado ausente: 34


,Nulos Treino,% Treino,Nulos Teste,% Teste
PoolQC,1453.0,99.52,1456.0,99.79
MiscFeature,1406.0,96.30,1408.0,96.50
Alley,1369.0,93.77,1352.0,92.67
Fence,1179.0,80.75,1169.0,80.12
MasVnrType,872.0,59.73,894.0,61.27
FireplaceQu,690.0,47.26,730.0,50.03
LotFrontage,259.0,17.74,227.0,15.56
GarageYrBlt,81.0,5.55,78.0,5.35
GarageFinish,81.0,5.55,78.0,5.35
GarageQual,81.0,5.55,78.0,5.35


### 2.2 Estratégia de Tratamento: Dois Grupos Distintos

Consultando o nosso [dicionario_dados_ptbr.md](dicionario_dados_ptbr.md), identificamos que os valores nulos se dividem em duas situações totalmente diferentes:

#### Grupo A: Ausência Estrutural ("A casa NÃO possui esse item")
Para essas colunas, o valor nulo não é um defeito de informação, mas sim a constatação de que o imóvel não tem o cômodo/recurso:
- **Piscina (`PoolQC`):** Se nulo, a casa não tem piscina. Preencheremos com `"Nenhum"`.
- **Recurso Diverso (`MiscFeature`):** Preencheremos com `"Nenhum"`.
- **Acesso por Beco (`Alley`):** Preencheremos com `"Nenhum"`.
- **Cerca (`Fence`):** Preencheremos com `"Nenhum"`.
- **Lareira (`FireplaceQu`):** Preencheremos com `"Nenhum"`.
- **Garagem (`GarageType`, `GarageFinish`, `GarageQual`, `GarageCond`):** Preencheremos com `"Nenhum"`.  
  *(Para as métricas numéricas `GarageCars` e `GarageArea`, preencheremos com `0`, e para `GarageYrBlt` preencheremos com `0` ou ano de construção da casa).*
- **Porão (`BsmtQual`, `BsmtCond`, `BsmtExposure`, `BsmtFinType1`, `BsmtFinType2`):** Preencheremos com `"Nenhum"`.  
  *(Para as áreas `BsmtFinSF1`, `BsmtFinSF2`, `BsmtUnfSF`, `TotalBsmtSF` e banheiros `BsmtFullBath`, preencheremos com `0`).*
- **Revestimento de Alvenaria (`MasVnrType`, `MasVnrArea`):** Preencheremos com `"Nenhum"` e `0`.

---

#### Grupo B: Dados Faltantes Reais (Omissão de Preenchimento)
Aqui a casa certamente possui o atributo, mas a informação não foi preenchida:
- **Frente do Lote (`LotFrontage`):** A casa tem frente para a rua. Como o tamanho da frente costuma ser padronizado pelo bairro, imputaremos pela **mediana do bairro (`Neighborhood`)** ao qual o imóvel pertence!
- **Variáveis Categóricas Esporádicas (`MSZoning`, `Utilities`, `Exterior1st`, `Exterior2nd`, `KitchenQual`, `Functional`, `SaleType`, `Electrical`):** Preencheremos com a **Moda** (o valor mais frequente da respectiva coluna).

In [3]:
# 1. Colunas onde NA significa "Não Possui" (Ausência Estrutural)
cols_nenhum = [
    'PoolQC', 'MiscFeature', 'Alley', 'Fence', 'FireplaceQu', 
    'GarageType', 'GarageFinish', 'GarageQual', 'GarageCond', 
    'BsmtQual', 'BsmtCond', 'BsmtExposure', 'BsmtFinType1', 'BsmtFinType2', 
    'MasVnrType'
]

cols_zero = [
    'GarageCars', 'GarageArea', 'GarageYrBlt', 
    'BsmtFinSF1', 'BsmtFinSF2', 'BsmtUnfSF', 'TotalBsmtSF', 
    'BsmtFullBath', 'BsmtHalfBath', 'MasVnrArea'
]

# 2. Colunas onde o nulo é omissão esporádica (Preenchimento por Moda)
cols_moda = [
    'MSZoning', 'Utilities', 'Exterior1st', 'Exterior2nd', 
    'KitchenQual', 'Functional', 'SaleType', 'Electrical'
]

# Criando cópias para teste da estratégia
df_tr_clean = df_train.copy()
df_te_clean = df_test.copy()

# Aplicando preenchimento estrutural
for df in [df_tr_clean, df_te_clean]:
    df[cols_nenhum] = df[cols_nenhum].fillna('None')
    df[cols_zero] = df[cols_zero].fillna(0)

# Preenchendo LotFrontage com a mediana do bairro (calculada sobre o treino para evitar data leakage)
medianas_bairro = df_tr_clean.groupby('Neighborhood')['LotFrontage'].median()
df_tr_clean['LotFrontage'] = df_tr_clean['LotFrontage'].fillna(df_tr_clean['Neighborhood'].map(medianas_bairro))
df_te_clean['LotFrontage'] = df_te_clean['LotFrontage'].fillna(df_te_clean['Neighborhood'].map(medianas_bairro))

# Preenchendo colunas esporádicas com a moda do treino
for col in cols_moda:
    moda_treino = df_tr_clean[col].mode()[0]
    df_tr_clean[col] = df_tr_clean[col].fillna(moda_treino)
    df_te_clean[col] = df_te_clean[col].fillna(moda_treino)

print("Imputação executada com sucesso!")

Imputação executada com sucesso!


In [4]:
# Verificação final: conferindo se resta algum valor nulo
nulos_restantes_treino = df_tr_clean.drop(columns=['SalePrice']).isnull().sum().sum()
nulos_restantes_teste = df_te_clean.isnull().sum().sum()

print("=" * 55)
print(" VERIFICAÇÃO PÓS-TRATAMENTO")
print("=" * 55)
print(f"Valores nulos restantes no Treino (X_train) : {nulos_restantes_treino}")
print(f"Valores nulos restantes no Teste  (X_test)  : {nulos_restantes_teste}")
print("=" * 55)

 VERIFICAÇÃO PÓS-TRATAMENTO
Valores nulos restantes no Treino (X_train) : 0
Valores nulos restantes no Teste  (X_test)  : 0


### 2.3 Conclusão da Etapa 2
Conseguimos eliminar 100% dos valores ausentes nos conjuntos de Treino e de Teste, respeitando a semântica real de cada variável:
1. Respeitamos a ausência de recursos (casas sem piscina, lareira, garagem ou porão receberam rótulos adequados e áreas zero).
2. Tratamos dados contínuos faltantes (`LotFrontage`) utilizando contexto geoespacial (mediana por bairro).
3. Evitamos **vazamento de dados (*data leakage*)**, calculando estatísticas (como medianas e modas) estritamente a partir do conjunto de treino e aplicando-as ao teste.

---

### Próximo Passo: Etapa 3 (Pré-processamento e Engenharia de Recursos)
Com os dados limpos, criaremos nosso primeiro script Python modular para:
- Codificar as variáveis categóricas (transformar textos em números que os algoritmos conseguem interpretar).
- Aplicar a transformação logarítmica na variável alvo `SalePrice`.
- Preparar os dados para o treinamento do modelo.